# Telco Churn: Modelos + Cross Validation + Hiperparametros
Notebook compacto para comparar 9 modelos con `RandomizedSearchCV` y evaluar el mejor de cada uno con `cross_val_score`.

In [ ]:
#mismo codigo que naive bayes

# Si falta alguna libreria, descomenta esta linea:
# %pip install -q pandas scikit-learn xgboost lightgbm

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.stats import randint, uniform

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis  #aqui se utiliza el lda y qda (diferente al otro codigo)
from sklearn.naive_bayes import GaussianNB

In [ ]:
# 1) Cargar Telco Churn desde URL
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

# 2) Limpieza minima
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna().copy()

# 3) Target + features
y = (df['Churn'] == 'Yes').astype(int)
X = df.drop(columns=['customerID', 'Churn'])

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()

preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

print(f'Datos: {X.shape[0]} filas, {X.shape[1]} columnas')
print(f'Churn rate: {y.mean():.2%}')

Datos: 7032 filas, 19 columnas
Churn rate: 26.58%


In [ ]:
models = {
    'knn': KNeighborsClassifier(),
    'dt': DecisionTreeClassifier(random_state=42),
    'random_forest': RandomForestClassifier(random_state=42),
    'logistic_regresion': LogisticRegression(max_iter=2000, random_state=42),
    'adaboost': AdaBoostClassifier(random_state=42),
    'xgboost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'gradientboostin': GradientBoostingClassifier(random_state=42),
    'litghxgboot': LGBMClassifier(random_state=42, verbose=-1),
    'mlp': MLPClassifier(max_iter=1200, random_state=42),

    # Nuevos modelos lda, qda y naive bayes
    'lda': LinearDiscriminantAnalysis(),
    'qda': QuadraticDiscriminantAnalysis(),
    'naive_bayes': GaussianNB(),
}

param_spaces = {
    'knn': {
        'model__n_neighbors': randint(3, 35),
        'model__weights': ['uniform', 'distance']
    },

    'dt': {
        'model__max_depth': randint(2, 20),
        'model__min_samples_split': randint(2, 20)
    },

    'random_forest': {
        'model__n_estimators': randint(100, 450),
        'model__max_depth': randint(3, 20),
        'model__min_samples_split': randint(2, 15)
    },

    'logistic_regresion': {
        'model__C': uniform(0.01, 8),
        'model__solver': ['lbfgs', 'liblinear']
    },

    'adaboost': {
        'model__n_estimators': randint(50, 300),
        'model__learning_rate': uniform(0.01, 1.0)
    },

    'xgboost': {
        'model__n_estimators': randint(80, 350),
        'model__max_depth': randint(2, 8),
        'model__learning_rate': uniform(0.02, 0.28),
        'model__subsample': uniform(0.6, 0.4)
    },

    'gradientboostin': {
        'model__n_estimators': randint(80, 320),
        'model__max_depth': randint(2, 7),
        'model__learning_rate': uniform(0.02, 0.28)
    },

    'litghxgboot': {
        'model__n_estimators': randint(80, 350),
        'model__num_leaves': randint(15, 120),
        'model__learning_rate': uniform(0.02, 0.28)
    },

    'mlp': {
        'model__hidden_layer_sizes': [(64,), (128,), (64, 32), (128, 64)],
        'model__alpha': uniform(1e-5, 1e-2),
        'model__learning_rate_init': uniform(1e-4, 5e-3)
    },

    # Nuevos espacios de hiperparámetros
    #aqui llamo a los nuevos modelos
    'lda': {
        'model__solver': ['svd', 'lsqr', 'eigen']   #solver es para crear esa linea de separacion
    },

    'qda': {
        'model__reg_param': uniform(0.0, 1.0)   #reg_param esel que regula esa linea
    },

    'naive_bayes': {
        'model__var_smoothing': uniform(1e-11, 1e-7)
    }
}

In [ ]:
# Tuneo + evaluacion final con varias metricas
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados = []

for name, model in models.items():
    pipe = Pipeline([('prep', preprocess), ('model', model)])

    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_spaces[name],
        n_iter=12,
        scoring='f1',
        cv=cv,
        n_jobs=-1,
        random_state=42
    )

    search.fit(X, y)
    best_model = search.best_estimator_

    f1_scores = cross_val_score(best_model, X, y, cv=cv, scoring='f1', n_jobs=-1)
    precision_scores = cross_val_score(best_model, X, y, cv=cv, scoring='precision', n_jobs=-1)
    recall_scores = cross_val_score(best_model, X, y, cv=cv, scoring='recall', n_jobs=-1)
    accuracy_scores = cross_val_score(best_model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    roc_auc_scores = cross_val_score(best_model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

    resultados.append({
        'modelo': name,
        'f1_mean': f1_scores.mean(),
        'f1_std': f1_scores.std(),
        'precision_mean': precision_scores.mean(),
        'recall_mean': recall_scores.mean(),
        'accuracy_mean': accuracy_scores.mean(),
        'roc_auc_mean': roc_auc_scores.mean(),
        'cv_error': 1 - f1_scores.mean(),
        'mejores_hiperparametros': str(search.best_params_)
    })

res_df = pd.DataFrame(resultados).sort_values('f1_mean', ascending=False).reset_index(drop=True)
res_df[['modelo', 'f1_mean', 'f1_std', 'precision_mean', 'recall_mean', 'accuracy_mean', 'roc_auc_mean', 'cv_error', 'mejores_hiperparametros']]

,modelo,f1_mean,f1_std,precision_mean,recall_mean,accuracy_mean,roc_auc_mean,cv_error,mejores_hiperparametros
0,qda,0.626952,0.011087,0.569364,0.697708,0.779437,0.826732,0.373048,{'model__reg_param': np.float64(0.708072577796...
1,logistic_regresion,0.601640,0.008883,0.660141,0.552697,0.805460,0.845160,0.398360,"{'model__C': np.float64(6.669541126403374), 'm..."
2,naive_bayes,0.594575,0.005685,0.458919,0.844306,0.693970,0.817535,0.405425,{'model__var_smoothing': np.float64(3.74640118...
3,knn,0.593684,0.013827,0.610379,0.578390,0.789533,0.830971,0.406316,"{'model__n_neighbors': 31, 'model__weights': '..."
4,lda,0.593111,0.015542,0.636566,0.555371,0.797496,0.836671,0.406889,{'model__solver': 'svd'}
5,xgboost,0.590227,0.007364,0.664087,0.531300,0.803896,0.845931,0.409773,{'model__learning_rate': np.float64(0.03821444...
6,adaboost,0.588231,0.010677,0.662520,0.529155,0.803042,0.847858,0.411769,{'model__learning_rate': np.float64(0.45583275...
7,gradientboostin,0.584002,0.008273,0.655856,0.526484,0.800625,0.843128,0.415998,{'model__learning_rate': np.float64(0.16398564...
8,random_forest,0.582764,0.011339,0.673805,0.513639,0.804464,0.845554,0.417236,"{'model__max_depth': 9, 'model__min_samples_sp..."
9,dt,0.581439,0.016284,0.546159,0.621713,0.762085,0.787027,0.418561,"{'model__max_depth': 2, 'model__min_samples_sp..."
